# ML-07 — Position baseline and review score

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w04_baseline_score.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Baseline construction

Pool training clicks/impressions by position band and smooth with 500 pseudo-impressions at global training CTR.

In [2]:
d,audit=study.read_data(); idx,counts=study.split_clients(d)
baseline=study.PositionBaseline().fit(d.iloc[idx['train']])
print(json.dumps(baseline.rates,indent=2))

{
  "1\u20133": 0.5854426154555458,
  "4\u201310": 0.32282305506884984,
  "11\u201320": 0.2936040639661615,
  "21\u201350": 0.15971536452501736,
  ">50": 0.02651122059100483
}


## Same-fold evaluation

Evaluate the baseline on unseen validation clients, then compare all methods on the same final test clients.

In [3]:
receipt_path=ROOT/'work/outputs/ctr_metrics.json'
if not receipt_path.exists():
    study.run()
r=json.loads(receipt_path.read_text())
print(pd.DataFrame(r['test']).T[['mae_pp','weighted_mae_pp','rmse_pp']].to_string())

                          mae_pp  weighted_mae_pp   rmse_pp
position_baseline       0.201323         0.204213  0.287002
random_forest           0.172950         0.172615  0.274226
hist_gradient_boosting  0.171077         0.174995  0.276305


## Score and reason codes

Rank positive exposure-scaled shortfalls against the selected reference; gaps are not predicted added clicks.

In [4]:
print('gap = impressions * max(predicted_ctr - observed_ctr, 0) / 100')
print('Rules: metadata/intent review for positions <=20; visibility review otherwise.')
print(json.dumps(r['recommendations'][:3],indent=2))

gap = impressions * max(predicted_ctr - observed_ctr, 0) / 100
Rules: metadata/intent review for positions <=20; visibility review otherwise.
[
  {
    "rank": 1,
    "review_id": "Review 01",
    "priority_score": 100.0,
    "reference_gap_clicks": 1403.3,
    "position_band": "4\u201310",
    "action": "Metadata and intent review",
    "reason_codes": [
      "below_grouped_ctr_reference",
      "visible_position_band",
      "word_count_missing",
      "recent_update_context"
    ],
    "evidence_level": "Higher measurement volume"
  },
  {
    "rank": 2,
    "review_id": "Review 02",
    "priority_score": 99.97,
    "reference_gap_clicks": 910.5,
    "position_band": "1\u20133",
    "action": "Metadata and intent review",
    "reason_codes": [
      "below_grouped_ctr_reference",
      "visible_position_band",
      "content_depth_context",
      "recent_update_context"
    ],
    "evidence_level": "Higher measurement volume"
  },
  {
    "rank": 3,
    "review_id": "Review 03",
  